# Notebook 2: IMST-Mamba 학습
Input: Triage 코드 + Notebook1 Output (처리된 데이터)

In [ ]:
# 1. GPU 확인
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU!')
!nvidia-smi | head -5

In [ ]:
# 2. 코드 복사
import os, shutil

def find_input_dir():
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'configs' in dirs and 'src' in dirs:
            return root
    return None

input_dir = find_input_dir()
assert input_dir, 'Triage 데이터셋을 찾을 수 없습니다'
dest = '/kaggle/working/Triage'
if os.path.exists(dest):
    shutil.rmtree(dest)
shutil.copytree(input_dir, dest)
print('코드 복사 완료:', os.listdir(dest))

In [ ]:
# 3. 처리된 데이터 복사 (Notebook1 Output - zip 압축 해제)
import os, shutil, zipfile, glob
from pathlib import Path

zip_path = '/kaggle/input/notebooks/parkyunhu1/notebook722c6297be/_output_.zip'
extract_dir = '/kaggle/working/nb1_output'

print('압축 해제 중...')
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)
print('압축 해제 완료')

candidates = glob.glob(f'{extract_dir}/**/processed', recursive=True)
print('발견:', candidates)
assert candidates, 'processed 폴더를 찾을 수 없습니다'

data_src = str(Path(candidates[0]).parent)
print('Data source:', data_src)

data_dest = '/kaggle/working/Triage/data'
if os.path.exists(data_dest):
    shutil.rmtree(data_dest)
shutil.copytree(data_src, data_dest)

for split in ['train', 'val', 'test']:
    n = len(list(Path(f'{data_dest}/processed/{split}').glob('*.pt')))
    print(f'{split}: {n} 파일')

In [ ]:
# 4. 패키지 설치
!pip install -q omegaconf lifelines statsmodels pyarrow

In [ ]:
# 5. Config 설정 (GPU T4 최적화)
import re, os
os.chdir('/kaggle/working/Triage')

with open('configs/base.yaml') as f:
    content = f.read()
content = re.sub(r'batch_size: \d+', 'batch_size: 16', content)
content = re.sub(r'max_epochs: \d+', 'max_epochs: 40', content)
content = re.sub(r'precision: \S+', 'precision: float16', content)
content = re.sub(r'early_stopping_patience: \d+', 'early_stopping_patience: 15', content)
content = re.sub(r'lr: [\d.e-]+', 'lr: 1.0e-4', content)
with open('configs/base.yaml', 'w') as f:
    f.write(content)

with open('configs/model_imst_mamba.yaml') as f:
    content = f.read()
content = re.sub(r'd_model: \d+', 'd_model: 256', content)
content = re.sub(r'd_state: \d+', 'd_state: 32', content)
content = re.sub(r'n_layers: \d+', 'n_layers: 3', content)
with open('configs/model_imst_mamba.yaml', 'w') as f:
    f.write(content)

print('Config 설정 완료')

In [ ]:
# 6. IMST-Mamba 학습 (seed 42, 123, 456)
import subprocess, sys, os

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/Triage'
os.makedirs('/kaggle/working/Triage/results/logs', exist_ok=True)

result = subprocess.run(
    [sys.executable, 'scripts/run_pipeline.py', '--stage', 'train',
     '--model', 'imst_mamba', '--seeds', '42', '123', '456'],
    env=env, capture_output=True, text=True,
    cwd='/kaggle/working/Triage'
)
print(result.stdout[-8000:] if len(result.stdout) > 8000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# 7. 결과 저장 확인
import os
from pathlib import Path

checkpoints = list(Path('/kaggle/working/Triage/results').glob('**/imst_mamba*_best.pt'))
print(f'저장된 체크포인트: {len(checkpoints)}개')
for ckpt in checkpoints:
    print(' -', ckpt)

# Output에 results 복사
import shutil
if os.path.exists('/kaggle/working/results'):
    shutil.rmtree('/kaggle/working/results')
shutil.copytree('/kaggle/working/Triage/results', '/kaggle/working/results')
print('Output에 저장 완료')